In [19]:
from torch.utils.data import Dataset, DataLoader
import tiktoken
class GP2Dataset(Dataset):
    
    def __init__(self, text, tokenizer, max_length, stride):
        
        self.input_ids = []
        self.target_ids = []
        ids = tokenizer.encode(text)
        
        for i in range(0, len(ids) - max_length, stride):
            input = ids[i: i + max_length]
            target = ids[i + 1 : i + max_length + 1 ]
            self.input_ids.append(input)
            self.target_ids.append(target)
            
    def __len__(self):
        return len(self.input_ids)        
    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]        
        
  


In [31]:
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch

with open("../data/mytext.txt", "r", encoding="utf-8") as f:
    text = f.read()


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]
    
    
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

Multi head attention

First, I define a self attention class
after that I implement casual attention class
finally, implement the multihead one

In [40]:
input = torch.randint(0, 50257, (1,6))  #creating an input seq of 6 tokens and 3 embedded size
print(input)

tensor([[46386, 17739, 22299, 17595, 11521, 35116]])


In [48]:
#first we create a word embedding and positional embedding so inputs ready for self attention
import torch.nn as nn

embed = nn.Embedding(50257, 3)
embeded = embed(input)

pos_embed = nn.Embedding(6, 3)
po = pos_embed(torch.arange(6))

inputs = embeded + po
print(inputs)

tensor([[[-0.4055, -1.0310, -1.4039],
         [-0.1980, -1.3975,  0.5651],
         [-0.9672, -1.6085,  1.1350],
         [-0.8814,  0.6229, -0.8604],
         [-1.2372, -0.0703,  2.8861],
         [ 0.6522,  0.3479,  2.0110]]], grad_fn=<AddBackward0>)


In [ ]:
#here I only considered the second input, and then finds the context vector for that.
inputs = inputs.squeeze(0)
print(inputs.shape)
query = inputs[1]
print(query.shape)
att_score_2 = torch.empty(inputs.shape[0])
print(att_score_2)    

for i, x_i in enumerate(inputs):
    att_score_2[i] = torch.dot(x_i, query)
    
att_score_2 = torch.softmax(att_score_2,dim=-1)
print(att_score_2) 

contex_vec = torch.zeros(query.shape)
for i, x in enumerate(inputs):
    
    contex_vec += att_score_2[i] * x
       
print(contex_vec)    

torch.Size([6, 3])
torch.Size([3])
tensor([-1.3879e-07,  1.9156e-42,  0.0000e+00,  1.9247e+00, -0.0000e+00,
         1.7663e+00])
tensor([0.0480, 0.2340, 0.5049, 0.0071, 0.1670, 0.0390],
       grad_fn=<SoftmaxBackward0>)
tensor([-0.7415, -1.1823,  1.1922], grad_fn=<AddBackward0>)


In [72]:
#we can overgeneralize it by using matrix multiplication 
att_score = inputs @ inputs.T
print(att_score.shape)

att_weights = torch.softmax(att_score, dim=-1)
print(att_weights)
print("All row sums:", att_weights.sum(dim=-1))

context_vec = att_weights @ inputs
print(context_vec)


torch.Size([6, 6])
tensor([[7.9722e-01, 6.7389e-02, 5.1410e-02, 8.1934e-02, 1.0051e-03, 1.0370e-03],
        [4.8008e-02, 2.3396e-01, 5.0489e-01, 7.1092e-03, 1.6698e-01, 3.9045e-02],
        [6.3802e-03, 8.7955e-02, 4.9621e-01, 1.3101e-03, 3.9610e-01, 1.2039e-02],
        [2.4606e-01, 2.9969e-02, 3.1702e-02, 6.5693e-01, 2.3235e-02, 1.2108e-02],
        [1.5832e-06, 3.6925e-04, 5.0279e-03, 1.2188e-05, 9.8719e-01, 7.4038e-03],
        [1.2860e-04, 6.7966e-03, 1.2030e-02, 4.9998e-04, 5.8283e-01, 3.9772e-01]],
       grad_fn=<SoftmaxBackward0>)
All row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       grad_fn=<SumBackward1>)
tensor([[-0.4591, -0.9475, -1.0883],
        [-0.7415, -1.1823,  1.1922],
        [-0.9833, -0.9505,  1.7703],
        [-0.7362,  0.0652, -0.7664],
        [-1.2215, -0.0755,  2.8700],
        [-0.4752,  0.0687,  2.4988]], grad_fn=<MmBackward0>)


In [ ]:
#now we can move forward and use trainable weights
#divide by sqrt of dimension since we want to keep the variance close to 1, it has a relation with variance.
#Also we dont want the values to be large
class SelfAttention(nn.Module):
    
    def __init__(self, d_in, d_out, qkv_bias =False):
        super().__init__()
        self.W_key = nn.Linear(d_in, d_out)
        self.W_query = nn.Linear(d_in, d_out)
        self.W_value = nn.Linear(d_in, d_out)
        
    def forward(self, x):
        keys = self.W_key(x)
        values = self.W_value(x)
        query = self.W_query(x)
        
        att_score = query @ keys.T
        att_weights = torch.softmax(att_score / keys.shape[-1] **0.5, dim=-1)
        print(keys.shape[-1])
        contex_vector = att_weights @ values
        
        return contex_vector            

torch.manual_seed(789)
sa_v2 = SelfAttention(3, 2)
print(sa_v2(inputs))



2
tensor([[-0.0064, -0.8191],
        [-0.0999, -0.7690],
        [-0.1574, -0.7411],
        [ 0.0078, -0.8352],
        [-0.1816, -0.7282],
        [-0.0650, -0.7823]], grad_fn=<MmBackward0>)


In [95]:
mask = torch.triu(torch.ones(6,6), diagonal=1)
print(mask)
scores = torch.randn(6,6)
scores.masked_fill_(mask.bool(),-torch.inf)
print(scores)




tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])
tensor([[-4.0313e-01,        -inf,        -inf,        -inf,        -inf,
                -inf],
        [-5.2222e-01,  7.9029e-02,        -inf,        -inf,        -inf,
                -inf],
        [-1.0060e+00, -5.3201e-02,  7.4095e-01,        -inf,        -inf,
                -inf],
        [-3.7250e-01, -1.7653e-02,  5.4686e-04, -8.0667e-01,        -inf,
                -inf],
        [-6.6460e-01,  4.0253e-01, -6.8343e-01, -9.2317e-01,  1.1656e+00,
                -inf],
        [-4.4840e-01, -7.2793e-01,  1.6059e+00,  4.1265e-01,  1.1509e+00,
          5.6637e-01]])


In [97]:
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout) # New
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # New

    def forward(self, x):
        b, num_tokens, d_in = x.shape # New batch dimension b
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2) # Changed transpose
        attn_scores.masked_fill_(  # New, _ ops are in-place
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights) # New

        context_vec = attn_weights @ values
        return context_vec

In [123]:
import torch.nn as nn

embed = nn.Embedding(50257, 3)
embeded = embed(input)

pos_embed = nn.Embedding(6, 3)
po = pos_embed(torch.arange(6))

inputs = embeded + po
print(inputs)
ca = CausalAttention(3,2,6,False)
contex = ca(inputs)
print(contex)

tensor([[[-1.6962,  0.1473, -1.9207],
         [ 0.3277,  1.3421, -0.1951],
         [ 2.0661,  0.9423,  0.7248],
         [ 0.1186, -0.8035, -1.0996],
         [-0.7977,  0.6866, -0.4235],
         [ 3.1245,  0.6576,  0.5280]]], grad_fn=<AddBackward0>)
tensor([[[-0.7457, -0.6637],
         [-0.6042, -0.5336],
         [-0.4001, -0.2921],
         [-0.2500, -0.0778],
         [-0.3073, -0.2013],
         [-0.2352, -0.0402]]], grad_fn=<UnsafeViewBackward0>)


In [133]:
x = torch.randn(2, 3)
print(x)
x.view(-1)
print(x)


tensor([[-1.0175, -1.3729,  0.2817],
        [ 0.9315, -0.3812,  1.2550]])
tensor([[-1.0175, -1.3729,  0.2817],
        [ 0.9315, -0.3812,  1.2550]])


In [134]:
import torch
import torch.nn as nn


class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

In [136]:
torch.manual_seed(123)

# Define the tensor with 3 rows and 6 columns
inputs = torch.tensor(
    [[0.43, 0.15, 0.89, 0.55, 0.87, 0.66],  # Row 1
     [0.57, 0.85, 0.64, 0.22, 0.58, 0.33],  # Row 2
     [0.77, 0.25, 0.10, 0.05, 0.80, 0.55]]  # Row 3
)

batch = torch.stack((inputs, inputs), dim=0)
print(batch)
print(batch.shape) 

batch_size, context_length, d_in = batch.shape
d_out = 6
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.4300, 0.1500, 0.8900, 0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400, 0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000, 0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900, 0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400, 0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000, 0.0500, 0.8000, 0.5500]]])
torch.Size([2, 3, 6])
tensor([[[ 0.1569, -0.0873,  0.0210,  0.0215, -0.3243, -0.2518],
         [ 0.1117, -0.0547,  0.0406, -0.0213, -0.3251, -0.2993],
         [ 0.1196, -0.0491,  0.0318, -0.0635, -0.2788, -0.2578]],

        [[ 0.1569, -0.0873,  0.0210,  0.0215, -0.3243, -0.2518],
         [ 0.1117, -0.0547,  0.0406, -0.0213, -0.3251, -0.2993],
         [ 0.1196, -0.0491,  0.0318, -0.0635, -0.2788, -0.2578]]],
       grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 3, 6])
